# company_hs_map_build_v1 - 기업-HS 매핑 정리 (STD-1)

**원본 (읽기만 함, 수정하지 않음)**: `korea_company_hscode_map`, `target_company_db` (hs_code가 숫자형이라 0이 빠진 코드가 섞여 있음)

**새로 만드는 테이블**: `kr_company_hs_map`

| 컬럼 | 내용 |
|---|---|
| ticker | 종목코드 6자리 (`A` 접두어 제거) |
| hs_code | **고친** HS 코드 (문자열, 앞자리 0 보존) |
| hs6 | 수출입 수집 단위 코드 (앞 6자리, 4자리 호 코드는 4자리 그대로) |
| in_collect_list | 현재 수출입 수집 목록(hs_code_map_v2)에 있는지 |
| hs_code_raw / fix_type / fix_note | 원래 값과 고친 이유 |

**고치는 코드 (2026-10-01 점검 결과)**

| 원래 | 고친 값 | 기업 | 사유 |
|---|---|---|---|
| 30341~30344 | 030341~030344 | 동원산업 | 앞자리 0 소실 (냉동 참치류) |
| 207 | 0207 | 해마로푸드서비스 | 앞자리 0 소실 (가금류 육) |
| 39022 | 390220 | DL | 소수점 입력으로 뒷자리 0 소실 (폴리이소부틸렌) |
| 19049 | 190490 | CJ제일제당 | 소수점 입력으로 뒷자리 0 소실 (곡물 조제식품) |
| 842951101 | 842951 | 현대건설기계 | 10자리 원본 확인 불가 → 6자리 사용 (사용자 확인) |

위 목록에 없는 홀수 자릿수 코드가 나오면 자동으로 고치지 않고 `unresolved`로 표시합니다.

## 1. 설정 및 접속

In [1]:
import sys
from pathlib import Path
from datetime import datetime, timedelta, timezone

import pandas as pd
from sqlalchemy import text

DATA_DIR = r"/DATA"

data_dir = Path(DATA_DIR)
if not data_dir.exists():
    data_dir = next((p / "DATA" for p in [Path.cwd(), *Path.cwd().parents] if (p / "DATA").exists()), None)
if data_dir is None:
    raise FileNotFoundError("DATA 폴더를 찾지 못했습니다. DATA_DIR 경로를 확인하세요.")
for p in (str(data_dir), str(data_dir.parent)):
    if p not in sys.path:
        sys.path.insert(0, p)

import config
from DATA.hs_code_map_v2 import HS_CODE_MAP

engine = config.get_engine(config.get_db_info())
KST = timezone(timedelta(hours=9))
T_MAP = "kr_company_hs_map"
pd.set_option("display.width", 220)
pd.set_option("display.max_rows", 200)
print("DB:", engine.url.database, "| 수집 목록 코드 수:", len(HS_CODE_MAP))

DB: investar | 수집 목록 코드 수: 501


## 2. 수정 규칙

In [2]:
# 원래 값(숫자형을 문자로 바꾼 것) → (고친 코드, 유형, 사유)
FIXES = {
    "30341": ("030341", "leading_zero", "앞자리 0 소실 - 날개다랑어(냉동)"),
    "30342": ("030342", "leading_zero", "앞자리 0 소실 - 황다랑어(냉동)"),
    "30343": ("030343", "leading_zero", "앞자리 0 소실 - 가다랑어(냉동)"),
    "30344": ("030344", "leading_zero", "앞자리 0 소실 - 눈다랑어(냉동)"),
    "207": ("0207", "leading_zero", "앞자리 0 소실 - 가금류 육"),
    "39022": ("390220", "trailing_zero", "소수점 입력(3902.20)으로 뒷자리 0 소실 - 폴리이소부틸렌"),
    "19049": ("190490", "trailing_zero", "소수점 입력(1904.90)으로 뒷자리 0 소실 - 곡물 조제식품"),
    "842951101": ("842951", "manual", "10자리 원본 확인 불가 → 6자리 사용 (사용자 확인 2026-10-01)"),
}


def fix_code(raw):
    """반환: (고친 코드, 유형, 사유)"""
    s = str(raw).strip()
    if s.endswith(".0"):
        s = s[:-2]
    if s in FIXES:
        return FIXES[s]
    if not s.isdigit():
        return s, "unresolved", "숫자가 아닌 코드"
    if len(s) % 2:
        return s, "unresolved", "홀수 자릿수 - 수정 규칙 없음 (확인 필요)"
    return s, "none", None


def norm_ticker(t):
    s = str(t or "").strip().upper()
    if s.startswith("A") and s[1:].isdigit():
        s = s[1:]
    return s.zfill(6) if s.isdigit() else s


def hs6_of(code_):
    return code_[:6] if len(code_) >= 6 else code_

## 3. 원본 읽기와 정리 (DB 변경 없음)

In [3]:
def read_source(table, tcol, ncol):
    with engine.connect() as cx:
        df = pd.read_sql(text(f"SELECT `{tcol}` AS ticker_raw, `{ncol}` AS company_name, "
                              f"CAST(hs_code AS CHAR) AS hs_code_raw FROM `{table}` WHERE hs_code IS NOT NULL"), cx)
    df["source_table"] = table
    return df


src = pd.concat([read_source("korea_company_hscode_map", "ticker", "Name"),
                 read_source("target_company_db", "Code", "Name")], ignore_index=True)
src["ticker"] = src["ticker_raw"].map(norm_ticker)
fixed = src["hs_code_raw"].map(fix_code)
src["hs_code"] = [f[0] for f in fixed]
src["fix_type"] = [f[1] for f in fixed]
src["fix_note"] = [f[2] for f in fixed]
src["hs6"] = src["hs_code"].map(hs6_of)
src["hs_digits"] = src["hs_code"].str.len()

# 같은 (종목, HS) 는 하나로: 출처 테이블 이름은 합쳐서 기록
grp = src.groupby(["ticker", "hs_code"], as_index=False)
m = grp.agg(ticker_raw=("ticker_raw", "first"), company_name=("company_name", "first"),
            hs_code_raw=("hs_code_raw", "first"), hs6=("hs6", "first"), hs_digits=("hs_digits", "first"),
            fix_type=("fix_type", "first"), fix_note=("fix_note", "first"),
            source_tables=("source_table", lambda s: ",".join(sorted(set(s)))))
m["hs_name"] = [HS_CODE_MAP.get(h) or HS_CODE_MAP.get(c) for h, c in zip(m["hs6"], m["hs_code"])]
m["in_collect_list"] = [int(h in HS_CODE_MAP or c in HS_CODE_MAP) for h, c in zip(m["hs6"], m["hs_code"])]

print(f"원본 {len(src)}행 → 정리 후 {len(m)}개 (종목 {m['ticker'].nunique()}개, HS {m['hs_code'].nunique()}개)")
print("\n수정 유형별 건수:")
print(m["fix_type"].value_counts().to_string())
print("\n고친 코드:")
print(m[m["fix_type"] != "none"][["ticker", "company_name", "hs_code_raw", "hs_code", "fix_type", "in_collect_list"]]
      .to_string(index=False))
unresolved = m[m["fix_type"] == "unresolved"]
if len(unresolved):
    print(f"\n[주의] 수정 규칙이 없는 코드 {len(unresolved)}개 - FIXES 에 추가가 필요합니다")

원본 1570행 → 정리 후 810개 (종목 533개, HS 501개)

수정 유형별 건수:
fix_type
none             802
leading_zero       5
trailing_zero      2
manual             1

고친 코드:
ticker company_name hs_code_raw hs_code      fix_type  in_collect_list
000210           DL       39022  390220 trailing_zero                1
006040         동원산업       30341  030341  leading_zero                1
006040         동원산업       30342  030342  leading_zero                1
006040         동원산업       30343  030343  leading_zero                1
006040         동원산업       30344  030344  leading_zero                1
097950       CJ제일제당       19049  190490 trailing_zero                1
220630     해마로푸드서비스         207    0207  leading_zero                1
267270       현대건설기계   842951101  842951        manual                1


## 4. 수출입 수집 목록에 없는 매핑 코드
매핑에는 있지만 `hs_code_map_v2`에 없어 **수출입 데이터가 수집되지 않는 코드**입니다. 필요하면 hs_code_map_v2에 추가하세요.

In [4]:
missing = (m[m["in_collect_list"] == 0]
           .groupby("hs6").agg(기업=("company_name", lambda s: ", ".join(sorted(set(s)))[:80]),
                               종목수=("ticker", "nunique"))
           .reset_index().sort_values("종목수", ascending=False))
print(f"수집 목록에 없는 hs6 코드: {len(missing)}개")
missing.head(50)

수집 목록에 없는 hs6 코드: 224개


,hs6,기업,종목수
201,8703,"네오티스, 대유에이텍, 모트렉스, 서진오토모티브, 에코플라스틱, 영신금속, 웹스, ...",12
163,847170,"GST, SFA반도체, 마이크로켄텍솔, 마이크로프랜드, 바른전자, 시스웍, 싸이맥스...",8
175,8504,"비에이치아이, 비츠로시스, 성문전자, 에스에너지, 일진전기, 제룡전기",6
33,3304,"CSA 코스믹, LG생활건강, SK바이오랜드, 씨티케이코스메틱스, 애경산업, 코스맥스",6
99,7204,"POSCO, 경남스틸, 대동스틸, 동일산업, 세아베스틸, 하이스틸",6
191,8535,"두산중공업, 우리기술, 이화전기, 일진전기, 일진파워, 한전기술",6
171,850212,"LIG넥스원, 에이테크솔루션, 조선선재, 화천기계, 휴림로봇",5
61,4819,"대양제지, 신풍제지, 태림포장, 한국수출포장공업, 한국제지",5
222,961610,"아모레퍼시픽, 에이블씨엔씨, 코리아나, 클리오, 토니모리",5
85,6202,"F&F, 신성통상, 신영와코루, 호전실업",4


## 5. `kr_company_hs_map` 저장

In [5]:
DDL = f"""
CREATE TABLE IF NOT EXISTS {T_MAP} (
    ticker             VARCHAR(20)  NOT NULL COMMENT '종목코드 6자리',
    hs_code            VARCHAR(10)  NOT NULL COMMENT '고친 HS 코드 (문자열)',
    ticker_raw         VARCHAR(20)  NULL     COMMENT '원본 종목코드 (A 접두어 포함)',
    company_name       VARCHAR(100) NULL,
    hs6                VARCHAR(6)   NOT NULL COMMENT '수출입 수집 단위 코드 (앞 6자리, 4자리는 그대로)',
    hs_digits          TINYINT      NOT NULL,
    hs_name            VARCHAR(500) NULL,
    in_collect_list    TINYINT      NOT NULL DEFAULT 0 COMMENT '1이면 hs_code_map_v2 수집 대상',
    hs_code_raw        VARCHAR(20)  NULL     COMMENT '원본 값 (숫자형에서 변환)',
    fix_type           VARCHAR(20)  NOT NULL COMMENT 'none | leading_zero | trailing_zero | manual | unresolved',
    fix_note           VARCHAR(255) NULL,
    source_tables      VARCHAR(100) NULL,
    is_active          TINYINT      NOT NULL DEFAULT 1 COMMENT '마지막 생성 때 원본에 있었으면 1',
    first_collected_at DATETIME     NOT NULL,
    last_collected_at  DATETIME     NOT NULL,
    created_at         TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP,
    updated_at         TIMESTAMP    NOT NULL DEFAULT CURRENT_TIMESTAMP ON UPDATE CURRENT_TIMESTAMP,
    PRIMARY KEY (ticker, hs_code),
    KEY idx_hs6 (hs6)
) ENGINE=InnoDB DEFAULT CHARSET=utf8mb4 COLLATE=utf8mb4_general_ci COMMENT='기업-HS 매핑 (STD-1, 앞자리 0 보정)'
"""
cols = ["ticker", "hs_code", "ticker_raw", "company_name", "hs6", "hs_digits", "hs_name", "in_collect_list",
        "hs_code_raw", "fix_type", "fix_note", "source_tables"]
ts = datetime.now(KST).replace(tzinfo=None, microsecond=0)
recs = m[cols].astype(object).where(m[cols].notna(), None).to_dict("records")
for r in recs:
    r["ts"] = ts
    r["hs_digits"] = int(r["hs_digits"]); r["in_collect_list"] = int(r["in_collect_list"])
upd = ", ".join(f"{c} = VALUES({c})" for c in cols[2:])
sql = text(f"""INSERT INTO {T_MAP} ({", ".join(cols)}, is_active, first_collected_at, last_collected_at)
               VALUES ({", ".join(":" + c for c in cols)}, 1, :ts, :ts)
               ON DUPLICATE KEY UPDATE {upd}, is_active = 1, last_collected_at = VALUES(last_collected_at)""")
with engine.begin() as cx:
    cx.execute(text(DDL))
    cx.execute(sql, recs)
    # 이번 원본에 없는 예전 행은 지우지 않고 비활성 표시
    n_off = cx.execute(text(f"UPDATE {T_MAP} SET is_active = 0 WHERE last_collected_at < :ts"), {"ts": ts}).rowcount
print(f"{T_MAP} 저장: {len(recs)}건, 비활성 처리 {n_off}건")

with engine.connect() as cx:
    chk = pd.read_sql(text(f"SELECT fix_type, COUNT(*) AS n, SUM(in_collect_list) AS in_list "
                           f"FROM {T_MAP} WHERE is_active = 1 GROUP BY fix_type"), cx)
chk

kr_company_hs_map 저장: 810건, 비활성 처리 0건


,fix_type,n,in_list
0,leading_zero,5,5.0
1,manual,1,1.0
2,none,802,435.0
3,trailing_zero,2,2.0
